In [1]:
import subprocess
import re
import pandas as pd
import shutil
from tqdm import tqdm
import openpyxl
import os
import gc

In [2]:
quartus_bin = r"C:\altera_standard\25.1std\quartus\bin64"

# Compile
subprocess.run(
    [f"{quartus_bin}\\quartus_sh.exe", "--flow", "compile", "generic_multiplier"],
        cwd=rf"C:\Users\Rafa\Desktop\HCR\generic_multiplier",
    check=True
)

# Timing
subprocess.run(
        [f"{quartus_bin}\\quartus_sta.exe", "-t", "script.tcl"],
        cwd=rf"C:\Users\Rafa\Desktop\HCR\generic_multiplier",
    check=True
)

CompletedProcess(args=['C:\\altera_standard\\25.1std\\quartus\\bin64\\quartus_sta.exe', '-t', 'script.tcl'], returncode=0)

## Diretorios

In [2]:
quartus_bin = r"C:\altera_standard\25.1std\quartus\bin64"
# base_dir = rf"C:\Users\Rafa\Desktop\HCR\generic_multiplier"
base_dir = rf"C:\Users\Rafa\Desktop\HCR\generic_multiplier_sqr"

n_values = [16, 24, 32, 48, 64, 96, 128, 256, 512] 


## Functions

In [3]:
# =========================
# altera n no VHDL
# =========================
def set_n(vhdl_file, n):
    with open(vhdl_file, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read()

    text = re.sub(
        r"n\s*:\s*integer\s*:=\s*\d+",
        f"n : integer := {n}",
        text
    )

    with open(vhdl_file, "w", encoding="utf-8") as f:
        f.write(text)


# =========================
# acha nome do projeto .qpf
# =========================
def find_project_name(project_dir):
    qpf_files = [f for f in os.listdir(project_dir) if f.endswith(".qpf")]

    if not qpf_files:
        raise FileNotFoundError(f"Nenhum arquivo .qpf encontrado em {project_dir}")

    return os.path.splitext(qpf_files[0])[0]


# =========================
# roda quartus
# =========================
def run_quartus(project_dir):
    shutil.rmtree(os.path.join(project_dir, "db"), ignore_errors=True)
    shutil.rmtree(os.path.join(project_dir, "incremental_db"), ignore_errors=True)

    project_name = find_project_name(project_dir)

    try:
        subprocess.run(
            [
                os.path.join(quartus_bin, "quartus_sh.exe"),
                "--flow",
                "compile",
                project_name
            ],
            cwd=project_dir,
            check=True,
            capture_output=True,
            text=True
        )

        subprocess.run(
            [
                os.path.join(quartus_bin, "quartus_sta.exe"),
                "-t",
                "script.tcl"
            ],
            cwd=project_dir,
            check=True,
            capture_output=True,
            text=True
        )
    except subprocess.CalledProcessError as e:
        raise RuntimeError(
            f"Quartus failed in {project_dir} with return code {e.returncode}\n"
            f"stdout:\n{e.stdout}\n"
            f"stderr:\n{e.stderr}"
        ) from e


# =========================
# parse dos paths
# =========================
def parse_paths(file):
    paths = []
    current = None

    with open(file, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:

            m = re.search(r"Path #\d+: Delay is\s+([\d\.]+)", line)
            if m:
                if current:
                    paths.append(current)

                current = {
                    "delay": float(m.group(1)),
                    "ic": None,
                    "cell": None
                }
                continue

            if current is None:
                continue

            m = re.search(r";\s*IC\s*;.*?;\s*\d+\s*;\s*([\d\.]+)", line)
            if m:
                current["ic"] = float(m.group(1))

            m = re.search(r";\s*Cell\s*;.*?;\s*\d+\s*;\s*([\d\.]+)", line)
            if m:
                current["cell"] = float(m.group(1))

    if current:
        paths.append(current)

    df = pd.DataFrame(paths)

    if df.empty:
        return df

    return df.dropna()


# =========================
# calcula métricas
# =========================
def compute_metrics(df):
    critical = df.loc[df["delay"].idxmax()]

    return {
        "Delay_CP": critical["delay"],
        "IC_CP": critical["ic"],
        "CELL_CP": critical["cell"],

        "Delay_MAX": df["delay"].max(),
        "IC_MAX": df["ic"].max(),
        "CELL_MAX": df["cell"].max(),

        "Delay_MEAN": df["delay"].mean(),
        "IC_MEAN": df["ic"].mean(),
        "CELL_MEAN": df["cell"].mean()
    }


# =========================
# extrai caminho crítico
# =========================
def extract_critical_path(file):
    delay = None
    ic = None
    cell = None
    from_node = None
    to_node = None

    with open(file, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:

            m = re.search(r"Delay is\s+([\d\.]+)", line)
            if m:
                delay = float(m.group(1))

            m = re.search(r";\s*From Node\s*;\s*(.*?)\s*;", line)
            if m:
                from_node = m.group(1)

            m = re.search(r";\s*To Node\s*;\s*(.*?)\s*;", line)
            if m:
                to_node = m.group(1)

            m = re.search(r";\s*IC\s*;.*?;\s*\d+\s*;\s*([\d\.]+)", line)
            if m:
                ic = float(m.group(1))

            m = re.search(r";\s*Cell\s*;.*?;\s*\d+\s*;\s*([\d\.]+)", line)
            if m:
                cell = float(m.group(1))

    return {
        "delay": delay,
        "ic": ic,
        "cell": cell,
        "from": from_node,
        "to": to_node
    }


In [5]:
# =========================
# loop principal
# =========================
resultados = []


project_dir = os.path.join(base_dir)
vhdl_file = os.path.join(project_dir, f"generic_multiplier.vhd")
report_file = os.path.join(project_dir, "io_paths.txt")
worst_file = os.path.join(project_dir, "worst_path.txt")

resultados_m = []

print(f"\n==============================")
print(f" Rodando projeto com vetores")
print(f"==============================")

for n in tqdm(n_values):

    df = None
    metrics = None
    cp = None

    try:
        print(f"\nRodando n = {n}")

        set_n(vhdl_file, n)
        run_quartus(project_dir)

        df = parse_paths(report_file)

        if df.empty:
            raise ValueError("Nenhum path encontrado em io_paths.txt")

        metrics = compute_metrics(df)
        cp = extract_critical_path(worst_file)

        linha = {
            "n": n,

            "Delay_CP": cp["delay"],
            "IC_CP": cp["ic"],
            "CELL_CP": cp["cell"],

            "From_Node": cp["from"],
            "To_Node": cp["to"],

            "Delay_MAX": metrics["Delay_MAX"],
            "IC_MAX": metrics["IC_MAX"],
            "CELL_MAX": metrics["CELL_MAX"],

            "Delay_MEAN": metrics["Delay_MEAN"],
            "IC_MEAN": metrics["IC_MEAN"],
            "CELL_MEAN": metrics["CELL_MEAN"],
        }

        resultados.append(linha)
        resultados_m.append(linha)

    except Exception as e:
        print(f"Erro em n={n}: {e}")
        continue

    finally:
        for name in ("df", "metrics", "cp"):
            globals().pop(name, None)
        gc.collect()

# salva um arquivo separado para cada número de vetores
df_m = pd.DataFrame(resultados_m).sort_values("n")

csv_m = os.path.join(base_dir, f"timing_n{n}.csv")
xlsx_m = os.path.join(base_dir, f"timing_n{n}.xlsx")

df_m.to_csv(csv_m, index=False)
df_m.to_excel(xlsx_m, index=False)

print(f"\nArquivos salvos para n = {n}:")
print(csv_m)
print(xlsx_m)


 Rodando projeto com vetores


  0%|          | 0/9 [00:00<?, ?it/s]


Rodando n = 16


 11%|█         | 1/9 [02:17<18:17, 137.22s/it]


Rodando n = 24


...


Rodando n = 256


 89%|████████▉ | 8/9 [20:30<02:49, 169.18s/it]


Rodando n = 512


100%|██████████| 9/9 [27:41<00:00, 184.61s/it]


Arquivos salvos para n = 512:
C:\Users\Rafa\Desktop\HCR\generic_multiplier_sqr\timing_n512.csv
C:\Users\Rafa\Desktop\HCR\generic_multiplier_sqr\timing_n512.xlsx


In [7]:
# =========================

# salva resultado geral
# =========================
final_df = pd.DataFrame(resultados).sort_values(["n"])

csv_final = os.path.join(base_dir, "timing_all_vectors.csv")
xlsx_final = os.path.join(base_dir, "timing_all_vectors.xlsx")

final_df.to_csv(csv_final, index=False)
final_df.to_excel(xlsx_final, index=False)

print("\nResultado final:")
print(final_df)

print("\nArquivos gerais salvos:")
print(csv_final)
print(xlsx_final)


Resultado final:
     n  Delay_CP   IC_CP  CELL_CP From_Node   To_Node  Delay_MAX  IC_MAX  \
0   16     2.615   0.896    1.719     a0[6]   sum[12]      2.615   0.896   
1   24     2.983   0.865    2.118    a1[11]    sum[3]      2.983   0.865   
2   32     3.674   1.217    2.457     a0[6]   sum[25]      3.674   1.558   
3   48     5.691   3.030    2.661     a0[3]   sum[46]      5.691   3.043   
4   64     8.022   4.974    3.048    a0[24]   sum[63]      8.022   5.023   
5   96    10.063   6.796    3.267    a1[49]   sum[78]     10.063   7.189   
6  128    11.696   7.404    4.292    a1[24]  sum[125]     11.696   7.454   
7  256    27.124  19.837    7.287    a0[31]  sum[250]     27.124  19.990   
8  512    49.771  40.002    9.769   a1[318]  sum[506]     49.771  41.679   

   CELL_MAX  Delay_MEAN    IC_MEAN  CELL_MEAN  
0     1.719    2.375549   0.703811   1.671738  
1     2.127    2.776375   0.689437   2.086937  
2     2.636    3.427321   0.916256   2.511065  
3     3.106    5.503734   2.8